# Notebook 03: Non-Linear Least Squares Inversion & Allan-Werle Deviation
Demonstrates from-scratch Levenberg-Marquardt fitting and stability characterization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tdlas.hitran_io import load_hitran_ch4_window
from tdlas.instrument import DeviceParams, simulate_scan
from tdlas.fit import levenberg_marquardt_fit
from tdlas.allan import compute_allan_deviation

lines = load_hitran_ch4_window(3017.0, 3019.0)
dev = DeviceParams.sample_random("DEV_FIT", seed=10)
rng = np.random.default_rng(10)
scan = simulate_scan(x_ppm=20.0, T_K=296.0, P_atm=1.0, device=dev, rng=rng, lines_data=lines)

t_norm = np.linspace(-0.5, 0.5, len(scan["voltage_v"]))
res = levenberg_marquardt_fit(scan["voltage_v"], scan["nu_axis_cm1"], t_norm, 296.0, 1.0, dev.path_length_cm, lines)
print(f"Ground Truth: 20.00 ppm | Fitted: {res["x_ppm"]:.3f} ppm (Converged: {res["converged"]})")

In [ ]:
# Allan Deviation on White Noise
noise_series = rng.normal(0.0, 0.1, 10000)
taus, oadev = compute_allan_deviation(noise_series, rate_hz=100.0)

plt.figure(figsize=(7, 4))
plt.loglog(taus, oadev, "o-", label="Allan Deviation σ_A(τ)")
plt.loglog(taus, oadev[0] * (taus / taus[0])**(-0.5), "k--", label="White Noise Reference (slope -0.5)")
plt.title("Allan-Werle Deviation")
plt.xlabel("Integration Time τ [s]")
plt.ylabel("Allan Deviation [ppm]")
plt.grid(True, which="both")
plt.legend()
plt.tight_layout()
plt.show()